# **Hospital Management System**

In [2]:
"""
Hospital Management System
============================

A professional, menu-driven Hospital Management System demonstrating
core Object-Oriented Programming principles (encapsulation, inheritance,
composition, abstraction and polymorphism).

The module is self-contained and relies only on the Python standard
library, so it can be run as-is in Google Colab, a local terminal, or
any standard Python 3.9+ interpreter.

Author: (Your Name)
License: MIT
"""

from __future__ import annotations

import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from typing import Dict, List, Optional


# ---------------------------------------------------------------------------
# Custom Exception Hierarchy
# ---------------------------------------------------------------------------
class HospitalError(Exception):
    """Base class for all hospital-related exceptions."""


class InvalidInputError(HospitalError):
    """Raised when user-supplied data fails validation."""


class PatientNotFoundError(HospitalError):
    """Raised when a requested patient does not exist."""


class DoctorNotFoundError(HospitalError):
    """Raised when a requested doctor does not exist."""


class DoctorUnavailableError(HospitalError):
    """Raised when a doctor has no matching availability slot."""


class AppointmentNotFoundError(HospitalError):
    """Raised when a requested appointment does not exist."""


class AppointmentConflictError(HospitalError):
    """Raised when a doctor already has an appointment at the requested slot."""


class InvalidAppointmentStateError(HospitalError):
    """Raised when an appointment transition is not permitted."""


class BedUnavailableError(HospitalError):
    """Raised when no beds are available in the requested ward."""


# ---------------------------------------------------------------------------
# Enumerations
# ---------------------------------------------------------------------------
class Department(Enum):
    """Hospital departments / doctor specializations."""

    GENERAL_MEDICINE = "General Medicine"
    CARDIOLOGY = "Cardiology"
    NEUROLOGY = "Neurology"
    ORTHOPEDICS = "Orthopedics"
    PEDIATRICS = "Pediatrics"
    DERMATOLOGY = "Dermatology"
    EMERGENCY = "Emergency"


class BloodGroup(Enum):
    A_POS = "A+"
    A_NEG = "A-"
    B_POS = "B+"
    B_NEG = "B-"
    AB_POS = "AB+"
    AB_NEG = "AB-"
    O_POS = "O+"
    O_NEG = "O-"
    UNKNOWN = "Unknown"


class AppointmentStatus(Enum):
    SCHEDULED = "Scheduled"
    COMPLETED = "Completed"
    CANCELLED = "Cancelled"
    NO_SHOW = "No-Show"


class AdmissionStatus(Enum):
    OUTPATIENT = "Outpatient"
    ADMITTED = "Admitted"
    DISCHARGED = "Discharged"


# ---------------------------------------------------------------------------
# Abstraction + Inheritance: Person hierarchy
# ---------------------------------------------------------------------------
class Person(ABC):
    """
    Abstract base class representing any individual in the hospital
    system (a patient or a doctor).

    Demonstrates encapsulation via private attributes exposed through
    validated properties, and abstraction via the ``get_role`` contract
    every subclass must fulfil.
    """

    def __init__(self, name: str, age: int, gender: str, phone: str) -> None:
        self._id: str = str(uuid.uuid4())[:8].upper()
        self.name = name
        self.age = age
        self.gender = gender
        self.phone = phone
        self._created_at: datetime = datetime.now()

    @property
    def id(self) -> str:
        return self._id

    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value: str) -> None:
        if not isinstance(value, str) or not value.strip():
            raise InvalidInputError("Name cannot be empty.")
        self._name = value.strip().title()

    @property
    def age(self) -> int:
        return self._age

    @age.setter
    def age(self, value: int) -> None:
        if not isinstance(value, int) or not (0 <= value <= 130):
            raise InvalidInputError(f"Invalid age: {value}")
        self._age = value

    @property
    def gender(self) -> str:
        return self._gender

    @gender.setter
    def gender(self, value: str) -> None:
        if not value or not value.strip():
            raise InvalidInputError("Gender cannot be empty.")
        self._gender = value.strip().title()

    @property
    def phone(self) -> str:
        return self._phone

    @phone.setter
    def phone(self, value: str) -> None:
        digits = (value or "").replace("-", "").replace(" ", "")
        if not digits.isdigit() or not (7 <= len(digits) <= 15):
            raise InvalidInputError(f"Invalid phone number: '{value}'")
        self._phone = value.strip()

    @abstractmethod
    def get_role(self) -> str:
        """Return a human-readable role label. Must be implemented by subclasses."""
        raise NotImplementedError

    def __str__(self) -> str:
        return f"{self.get_role()} | {self.name} (ID: {self.id}, Age: {self.age})"

    def __repr__(self) -> str:
        return f"<{self.__class__.__name__} id={self.id!r} name={self.name!r}>"


class Patient(Person):
    """A hospital patient with a medical record and admission status."""

    def __init__(
        self,
        name: str,
        age: int,
        gender: str,
        phone: str,
        blood_group: BloodGroup = BloodGroup.UNKNOWN,
    ) -> None:
        super().__init__(name, age, gender, phone)
        self.blood_group = blood_group
        self._medical_history: List[str] = []
        self._admission_status: AdmissionStatus = AdmissionStatus.OUTPATIENT
        self._ward: Optional["Ward"] = None
        self._appointments: List["Appointment"] = []
        self._bills: List["Bill"] = []

    def get_role(self) -> str:
        return "Patient"

    @property
    def medical_history(self) -> List[str]:
        return list(self._medical_history)

    def add_medical_note(self, note: str) -> None:
        if not note.strip():
            raise InvalidInputError("Medical note cannot be empty.")
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        self._medical_history.append(f"[{timestamp}] {note.strip()}")

    @property
    def admission_status(self) -> AdmissionStatus:
        return self._admission_status

    @property
    def ward(self) -> Optional["Ward"]:
        return self._ward

    def _admit(self, ward: "Ward") -> None:
        self._admission_status = AdmissionStatus.ADMITTED
        self._ward = ward

    def _discharge(self) -> None:
        self._admission_status = AdmissionStatus.DISCHARGED
        self._ward = None

    def _attach_appointment(self, appointment: "Appointment") -> None:
        self._appointments.append(appointment)

    @property
    def appointments(self) -> List["Appointment"]:
        return list(self._appointments)

    def _attach_bill(self, bill: "Bill") -> None:
        self._bills.append(bill)

    @property
    def bills(self) -> List["Bill"]:
        return list(self._bills)

    @property
    def total_due(self) -> float:
        return round(sum(b.balance for b in self._bills), 2)


class Doctor(Person):
    """A hospital doctor with a specialization and appointment schedule."""

    def __init__(
        self,
        name: str,
        age: int,
        gender: str,
        phone: str,
        department: Department,
        consultation_fee: float = 500.0,
    ) -> None:
        super().__init__(name, age, gender, phone)
        self.department = department
        if consultation_fee < 0:
            raise InvalidInputError("Consultation fee cannot be negative.")
        self.consultation_fee = consultation_fee
        self._schedule: Dict[str, "Appointment"] = {}  # slot key -> Appointment

    def get_role(self) -> str:
        return "Doctor"

    @staticmethod
    def _slot_key(slot: datetime) -> str:
        return slot.strftime("%Y-%m-%d %H:%M")

    def is_free_at(self, slot: datetime) -> bool:
        return self._slot_key(slot) not in self._schedule

    def _book_slot(self, appointment: "Appointment") -> None:
        key = self._slot_key(appointment.slot)
        if key in self._schedule:
            raise AppointmentConflictError(
                f"Dr. {self.name} already has an appointment at {key}."
            )
        self._schedule[key] = appointment

    def _free_slot(self, slot: datetime) -> None:
        self._schedule.pop(self._slot_key(slot), None)

    @property
    def upcoming_appointments(self) -> List["Appointment"]:
        return sorted(
            (a for a in self._schedule.values() if a.status == AppointmentStatus.SCHEDULED),
            key=lambda a: a.slot,
        )


# ---------------------------------------------------------------------------
# Ward (bed inventory — encapsulation)
# ---------------------------------------------------------------------------
class Ward:
    """Represents a hospital ward with a fixed bed capacity."""

    def __init__(self, name: str, total_beds: int):
        if total_beds < 1:
            raise InvalidInputError("A ward must have at least one bed.")
        self.name = name
        self._total_beds = total_beds
        self._occupied_beds = 0

    @property
    def total_beds(self) -> int:
        return self._total_beds

    @property
    def available_beds(self) -> int:
        return self._total_beds - self._occupied_beds

    def _occupy_bed(self) -> None:
        if self.available_beds <= 0:
            raise BedUnavailableError(f"No available beds in ward '{self.name}'.")
        self._occupied_beds += 1

    def _release_bed(self) -> None:
        if self._occupied_beds > 0:
            self._occupied_beds -= 1

    def __str__(self) -> str:
        return f"Ward '{self.name}' ({self.available_beds}/{self.total_beds} beds free)"


# ---------------------------------------------------------------------------
# Appointment (transaction record)
# ---------------------------------------------------------------------------
@dataclass
class Appointment:
    """A scheduled consultation between a patient and a doctor."""

    appointment_id: str
    patient_id: str
    doctor_id: str
    slot: datetime
    reason: str
    status: AppointmentStatus = AppointmentStatus.SCHEDULED
    diagnosis: Optional[str] = None

    def __str__(self) -> str:
        return (
            f"Appt#{self.appointment_id} | {self.slot:%Y-%m-%d %H:%M} | "
            f"Reason: {self.reason} | Status: {self.status.value}"
        )


# ---------------------------------------------------------------------------
# Billing (composition inside Patient interactions)
# ---------------------------------------------------------------------------
@dataclass
class BillItem:
    description: str
    amount: float


class Bill:
    """An itemized bill accumulated for a patient."""

    def __init__(self, patient_id: str) -> None:
        self.bill_id = str(uuid.uuid4())[:8].upper()
        self.patient_id = patient_id
        self._items: List[BillItem] = []
        self._amount_paid: float = 0.0
        self.created_at = datetime.now()

    @property
    def items(self) -> List[BillItem]:
        return list(self._items)

    def add_item(self, description: str, amount: float) -> None:
        if amount <= 0:
            raise InvalidInputError("Bill item amount must be positive.")
        self._items.append(BillItem(description.strip(), round(amount, 2)))

    @property
    def total(self) -> float:
        return round(sum(item.amount for item in self._items), 2)

    @property
    def amount_paid(self) -> float:
        return round(self._amount_paid, 2)

    @property
    def balance(self) -> float:
        return round(self.total - self._amount_paid, 2)

    def pay(self, amount: float) -> float:
        if amount <= 0:
            raise InvalidInputError("Payment amount must be positive.")
        self._amount_paid = round(min(self.total, self._amount_paid + amount), 2)
        return self.balance

    def __str__(self) -> str:
        return f"Bill#{self.bill_id} | Total: ${self.total:.2f} | Balance: ${self.balance:.2f}"


# ---------------------------------------------------------------------------
# Hospital (Composition root / Facade over the whole domain)
# ---------------------------------------------------------------------------
class Hospital:
    """
    Central coordinator that *composes* patients, doctors, wards, and
    appointments. Cross-entity business rules (scheduling, admission,
    billing) live here so individual entities stay focused on their own
    state (single-responsibility principle).
    """

    def __init__(self, name: str) -> None:
        self.name = name
        self._patients: Dict[str, Patient] = {}
        self._doctors: Dict[str, Doctor] = {}
        self._wards: Dict[str, Ward] = {}
        self._appointments: Dict[str, Appointment] = {}
        self._bills: Dict[str, Bill] = {}

    # -- Registration ------------------------------------------------
    def register_patient(
        self,
        name: str,
        age: int,
        gender: str,
        phone: str,
        blood_group: BloodGroup = BloodGroup.UNKNOWN,
    ) -> Patient:
        patient = Patient(name, age, gender, phone, blood_group)
        self._patients[patient.id] = patient
        return patient

    def get_patient(self, patient_id: str) -> Patient:
        patient = self._patients.get(patient_id.strip().upper())
        if patient is None:
            raise PatientNotFoundError(f"No patient found with ID '{patient_id}'.")
        return patient

    @property
    def patients(self) -> List[Patient]:
        return sorted(self._patients.values(), key=lambda p: p.name)

    def register_doctor(
        self,
        name: str,
        age: int,
        gender: str,
        phone: str,
        department: Department,
        consultation_fee: float = 500.0,
    ) -> Doctor:
        doctor = Doctor(name, age, gender, phone, department, consultation_fee)
        self._doctors[doctor.id] = doctor
        return doctor

    def get_doctor(self, doctor_id: str) -> Doctor:
        doctor = self._doctors.get(doctor_id.strip().upper())
        if doctor is None:
            raise DoctorNotFoundError(f"No doctor found with ID '{doctor_id}'.")
        return doctor

    @property
    def doctors(self) -> List[Doctor]:
        return sorted(self._doctors.values(), key=lambda d: d.name)

    def doctors_by_department(self, department: Department) -> List[Doctor]:
        return [d for d in self._doctors.values() if d.department == department]

    # -- Wards -------------------------------------------------------
    def add_ward(self, name: str, total_beds: int) -> Ward:
        ward = Ward(name, total_beds)
        self._wards[ward.name] = ward
        return ward

    def get_ward(self, name: str) -> Ward:
        ward = self._wards.get(name.strip())
        if ward is None:
            raise HospitalError(f"No ward found named '{name}'.")
        return ward

    @property
    def wards(self) -> List[Ward]:
        return list(self._wards.values())

    def admit_patient(self, patient_id: str, ward_name: str) -> None:
        patient = self.get_patient(patient_id)
        ward = self.get_ward(ward_name)
        if patient.admission_status == AdmissionStatus.ADMITTED:
            raise InvalidAppointmentStateError(f"{patient.name} is already admitted.")
        ward._occupy_bed()
        patient._admit(ward)

    def discharge_patient(self, patient_id: str) -> None:
        patient = self.get_patient(patient_id)
        if patient.admission_status != AdmissionStatus.ADMITTED:
            raise InvalidAppointmentStateError(f"{patient.name} is not currently admitted.")
        patient.ward._release_bed()
        patient._discharge()

    # -- Appointments --------------------------------------------------
    def book_appointment(
        self, patient_id: str, doctor_id: str, slot: datetime, reason: str
    ) -> Appointment:
        patient = self.get_patient(patient_id)
        doctor = self.get_doctor(doctor_id)
        if slot < datetime.now():
            raise InvalidInputError("Appointment slot cannot be in the past.")
        if not reason.strip():
            raise InvalidInputError("A reason for the visit is required.")
        if not doctor.is_free_at(slot):
            raise DoctorUnavailableError(
                f"Dr. {doctor.name} is not available at {slot:%Y-%m-%d %H:%M}."
            )

        appointment = Appointment(
            appointment_id=str(uuid.uuid4())[:8].upper(),
            patient_id=patient.id,
            doctor_id=doctor.id,
            slot=slot,
            reason=reason.strip(),
        )
        doctor._book_slot(appointment)
        patient._attach_appointment(appointment)
        self._appointments[appointment.appointment_id] = appointment
        return appointment

    def get_appointment(self, appointment_id: str) -> Appointment:
        appointment = self._appointments.get(appointment_id.strip().upper())
        if appointment is None:
            raise AppointmentNotFoundError(f"No appointment found with ID '{appointment_id}'.")
        return appointment

    def complete_appointment(self, appointment_id: str, diagnosis: str) -> Bill:
        appointment = self.get_appointment(appointment_id)
        if appointment.status != AppointmentStatus.SCHEDULED:
            raise InvalidAppointmentStateError(
                f"Appointment '{appointment_id}' is not in a schedulable state "
                f"(current status: {appointment.status.value})."
            )
        doctor = self.get_doctor(appointment.doctor_id)
        patient = self.get_patient(appointment.patient_id)

        appointment.status = AppointmentStatus.COMPLETED
        appointment.diagnosis = diagnosis.strip() or "N/A"
        patient.add_medical_note(f"Consulted Dr. {doctor.name}: {appointment.diagnosis}")
        doctor._free_slot(appointment.slot)

        bill = self._get_or_create_bill(patient.id)
        bill.add_item(f"Consultation with Dr. {doctor.name}", doctor.consultation_fee)
        return bill

    def cancel_appointment(self, appointment_id: str) -> None:
        appointment = self.get_appointment(appointment_id)
        if appointment.status != AppointmentStatus.SCHEDULED:
            raise InvalidAppointmentStateError(
                f"Only scheduled appointments can be cancelled "
                f"(current status: {appointment.status.value})."
            )
        doctor = self.get_doctor(appointment.doctor_id)
        appointment.status = AppointmentStatus.CANCELLED
        doctor._free_slot(appointment.slot)

    # -- Billing --------------------------------------------------------
    def _get_or_create_bill(self, patient_id: str) -> Bill:
        for bill in self._bills.values():
            if bill.patient_id == patient_id and bill.balance > 0:
                return bill
        patient = self.get_patient(patient_id)
        bill = Bill(patient_id)
        self._bills[bill.bill_id] = bill
        patient._attach_bill(bill)
        return bill

    def get_bill(self, bill_id: str) -> Bill:
        bill = self._bills.get(bill_id.strip().upper())
        if bill is None:
            raise HospitalError(f"No bill found with ID '{bill_id}'.")
        return bill

    def pay_bill(self, bill_id: str, amount: float) -> float:
        bill = self.get_bill(bill_id)
        return bill.pay(amount)

    # -- Reporting --------------------------------------------------------
    def statistics(self) -> Dict[str, int]:
        admitted = sum(
            1 for p in self._patients.values() if p.admission_status == AdmissionStatus.ADMITTED
        )
        scheduled = sum(
            1 for a in self._appointments.values() if a.status == AppointmentStatus.SCHEDULED
        )
        total_beds = sum(w.total_beds for w in self._wards.values())
        free_beds = sum(w.available_beds for w in self._wards.values())
        return {
            "patients": len(self._patients),
            "doctors": len(self._doctors),
            "admitted_patients": admitted,
            "scheduled_appointments": scheduled,
            "total_beds": total_beds,
            "available_beds": free_beds,
        }


# ---------------------------------------------------------------------------
# Command Line Interface
# ---------------------------------------------------------------------------
class HospitalCLI:
    """Menu-driven console front-end for the :class:`Hospital` domain model."""

    def __init__(self) -> None:
        self.hospital = Hospital("Sunrise General Hospital")
        self._seed_demo_data()

    def _seed_demo_data(self) -> None:
        self.hospital.register_doctor(
            "Emily Carter", 42, "Female", "9998887771", Department.CARDIOLOGY, 800.0
        )
        self.hospital.register_doctor(
            "Michael Chen", 38, "Male", "9998887772", Department.GENERAL_MEDICINE, 500.0
        )
        self.hospital.add_ward("General Ward", 10)
        self.hospital.add_ward("ICU", 4)

    def run(self) -> None:
        print(f"\nWelcome to {self.hospital.name} Management System")
        actions = {
            "1": self._register_patient,
            "2": self._register_doctor,
            "3": self._list_doctors,
            "4": self._book_appointment,
            "5": self._complete_appointment,
            "6": self._cancel_appointment,
            "7": self._admit_patient,
            "8": self._discharge_patient,
            "9": self._view_patient,
            "10": self._pay_bill,
            "11": self._statistics,
        }
        while True:
            self._print_menu()
            choice = input("Enter your choice: ").strip()
            if choice == "0":
                print("Goodbye!")
                break
            action = actions.get(choice)
            if action is None:
                print("Invalid choice. Please select a valid menu option.\n")
                continue
            try:
                action()
            except HospitalError as exc:
                print(f"\n[Error] {exc}\n")
            except ValueError:
                print("\n[Error] Please enter a valid value (check numbers/dates).\n")

    @staticmethod
    def _print_menu() -> None:
        print(
            "\n"
            "==================== MAIN MENU ====================\n"
            " 1. Register Patient\n"
            " 2. Register Doctor\n"
            " 3. List Doctors\n"
            " 4. Book Appointment\n"
            " 5. Complete Appointment (add diagnosis + bill)\n"
            " 6. Cancel Appointment\n"
            " 7. Admit Patient\n"
            " 8. Discharge Patient\n"
            " 9. View Patient Details\n"
            "10. Pay Bill\n"
            "11. Hospital Statistics\n"
            " 0. Exit\n"
            "===================================================="
        )

    # -- Menu actions -----------------------------------------------------
    def _register_patient(self) -> None:
        print("\n-- Register New Patient --")
        name = input("Full name: ").strip()
        age = int(input("Age: ").strip())
        gender = input("Gender: ").strip()
        phone = input("Phone: ").strip()
        print("Blood groups:", ", ".join(b.value for b in BloodGroup))
        bg_input = input("Blood group [Unknown]: ").strip() or "Unknown"
        blood_group = next((b for b in BloodGroup if b.value.lower() == bg_input.lower()), BloodGroup.UNKNOWN)
        patient = self.hospital.register_patient(name, age, gender, phone, blood_group)
        print(f"\nSuccess: Registered {patient}. Patient ID: {patient.id}")

    def _register_doctor(self) -> None:
        print("\n-- Register New Doctor --")
        name = input("Full name: ").strip()
        age = int(input("Age: ").strip())
        gender = input("Gender: ").strip()
        phone = input("Phone: ").strip()
        print("Departments:", ", ".join(d.value for d in Department))
        dept_input = input("Department: ").strip().title()
        department = next(
            (d for d in Department if d.value.lower() == dept_input.lower()),
            Department.GENERAL_MEDICINE,
        )
        fee = float(input("Consultation fee [500]: ").strip() or "500")
        doctor = self.hospital.register_doctor(name, age, gender, phone, department, fee)
        print(f"\nSuccess: Registered {doctor}. Doctor ID: {doctor.id}")

    def _list_doctors(self) -> None:
        doctors = self.hospital.doctors
        if not doctors:
            print("\nNo doctors registered yet.")
            return
        print(f"\n-- Doctors ({len(doctors)}) --")
        for doctor in doctors:
            print(f"  - {doctor} | {doctor.department.value} | Fee: ${doctor.consultation_fee:.2f}")

    def _book_appointment(self) -> None:
        print("\n-- Book Appointment --")
        patient_id = input("Patient ID: ").strip()
        doctor_id = input("Doctor ID: ").strip()
        date_str = input("Date & time (YYYY-MM-DD HH:MM): ").strip()
        slot = datetime.strptime(date_str, "%Y-%m-%d %H:%M")
        reason = input("Reason for visit: ").strip()
        appointment = self.hospital.book_appointment(patient_id, doctor_id, slot, reason)
        print(f"\nSuccess: Appointment booked. {appointment}")

    def _complete_appointment(self) -> None:
        print("\n-- Complete Appointment --")
        appointment_id = input("Appointment ID: ").strip()
        diagnosis = input("Diagnosis / notes: ").strip()
        bill = self.hospital.complete_appointment(appointment_id, diagnosis)
        print(f"\nSuccess: Appointment completed. {bill}")

    def _cancel_appointment(self) -> None:
        appointment_id = input("\nAppointment ID to cancel: ").strip()
        self.hospital.cancel_appointment(appointment_id)
        print("\nSuccess: Appointment cancelled.")

    def _admit_patient(self) -> None:
        print("\n-- Admit Patient --")
        patient_id = input("Patient ID: ").strip()
        print("Wards:")
        for ward in self.hospital.wards:
            print(f"  - {ward}")
        ward_name = input("Ward name: ").strip()
        self.hospital.admit_patient(patient_id, ward_name)
        print("\nSuccess: Patient admitted.")

    def _discharge_patient(self) -> None:
        patient_id = input("\nPatient ID to discharge: ").strip()
        self.hospital.discharge_patient(patient_id)
        print("\nSuccess: Patient discharged.")

    def _view_patient(self) -> None:
        patient_id = input("\nPatient ID: ").strip()
        patient = self.hospital.get_patient(patient_id)
        print(f"\n{patient}")
        print(f"Blood group: {patient.blood_group.value}")
        print(f"Admission status: {patient.admission_status.value}")
        if patient.ward:
            print(f"Ward: {patient.ward.name}")
        print(f"Total outstanding balance: ${patient.total_due:.2f}")
        if patient.appointments:
            print("Appointments:")
            for appt in patient.appointments:
                print(f"  - {appt}")
        if patient.medical_history:
            print("Medical history:")
            for note in patient.medical_history:
                print(f"  - {note}")

    def _pay_bill(self) -> None:
        bill_id = input("\nBill ID: ").strip()
        bill = self.hospital.get_bill(bill_id)
        print(f"Current balance: ${bill.balance:.2f}")
        if bill.balance <= 0:
            print("This bill is already fully paid.")
            return
        amount = float(input("Payment amount: ").strip())
        remaining = self.hospital.pay_bill(bill_id, amount)
        print(f"Payment accepted. Remaining balance: ${remaining:.2f}")

    def _statistics(self) -> None:
        stats = self.hospital.statistics()
        print("\n-- Hospital Statistics --")
        for key, value in stats.items():
            print(f"  {key.replace('_', ' ').title()}: {value}")


def main() -> None:
    """Entry point for running the system as a script or in a notebook cell."""
    cli = HospitalCLI()
    cli.run()


if __name__ == "__main__":
    main()


Welcome to Sunrise General Hospital Management System

==================== MAIN MENU ====================
 1. Register Patient
 2. Register Doctor
 3. List Doctors
 4. Book Appointment
 5. Complete Appointment (add diagnosis + bill)
 6. Cancel Appointment
 7. Admit Patient
 8. Discharge Patient
 9. View Patient Details
10. Pay Bill
11. Hospital Statistics
 0. Exit
Enter your choice: 1

-- Register New Patient --
Full name: Ram
Age: 42
Gender: Male
Phone: 998642635
Blood groups: A+, A-, B+, B-, AB+, AB-, O+, O-, Unknown
Blood group [Unknown]: B-

Success: Registered Patient | Ram (ID: 96D811E3, Age: 42). Patient ID: 96D811E3

==================== MAIN MENU ====================
 1. Register Patient
 2. Register Doctor
 3. List Doctors
 4. Book Appointment
 5. Complete Appointment (add diagnosis + bill)
 6. Cancel Appointment
 7. Admit Patient
 8. Discharge Patient
 9. View Patient Details
10. Pay Bill
11. Hospital Statistics
 0. Exit
Enter your choice: 1

-- Register New Patient --
Ful